In [1]:
!nvidia-smi

Sun Mar 15 16:52:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%cd /content/drive/MyDrive/5398/LLaMA-Factory

/content/drive/MyDrive/5398/LLaMA-Factory


In [4]:
!python -m pip install -e .
!python -m pip list | grep llamafactory
!pip install -e .[torch,bitsandbytes]
!pip install rouge
#run this code once, do not rerun this code after restarting the session

Obtaining file:///content/drive/MyDrive/5398/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.8/109.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 127.3 MB/s eta 0:00:00
  Building editable for llamafactory (pyproject.tom

In [4]:
import platform
platform.system(), platform.release()

('Linux', '6.6.113+')

In [5]:
from google.colab import drive
import huggingface_hub
from google.colab import userdata
import pandas as pd
import json
import os
import torch
import time
from rouge import Rouge

In [6]:
import huggingface_hub
from google.colab import userdata
HF_t = userdata.get('HF_TOKEN')
huggingface_hub.login(HF_t)

In [ ]:
'''
%cd /content/drive/MyDrive/5398
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]
'''

'\n%cd /content/drive/MyDrive/5398\n%rm -rf LLaMA-Factory\n!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git\n%cd LLaMA-Factory\n%ls\n!pip install -e .[torch,bitsandbytes]\n'

In [7]:
import time
from rouge import Rouge

In [8]:
%cd /content/drive/MyDrive/5398/LLaMA-Factory

/content/drive/MyDrive/5398/LLaMA-Factory


In [10]:
%cd /content/drive/MyDrive/5398/LLaMA-Factory
!python -m pip install -e .
#after running this cell, restart the runtime. Do not rerun this cell after restarting

/content/drive/MyDrive/5398/LLaMA-Factory
Obtaining file:///content/drive/MyDrive/5398/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.5.dev0-py3-none-any.whl size=26999 sha256=92f8249a907c6938beda7479f7d1135c5b098365aac07733fcd12af9bffb5a67
  Stored in directory: /tmp/pip-ephem-wheel-cache-zazl0z_x/wheels/4a/98/83/0ecf5c7c171d2c5f1089381f844e4c9bc1ff306066f8b26dff
Successfully built llamafactory
  Attempting uninstall: llamafactory
    Found existing installation: llamafactory 0.9.5.dev0
    Uninstalling llamafactory-0.9.5.dev0:
      Successfully uninstalled llamafactory-0.9.5.dev0


# **1. Llama3.1 8b**

**1.1 Fine Tune**

In [9]:
args = dict(
  stage="sft",
  do_train=True,
  model_name_or_path="meta-llama/Llama-3.1-8B",
  dataset="df_train_system",
  template="llama3",
  finetuning_type="lora",
  lora_target="q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
  output_dir="llama3.1_lora",
  per_device_train_batch_size=2,
  gradient_accumulation_steps=16,
  lr_scheduler_type="cosine",
  logging_steps=5,
  warmup_ratio=0.1,
  save_steps=1000,
  learning_rate=5e-5,
  num_train_epochs=3.0,
  max_grad_norm=1.0,
  loraplus_lr_ratio=16.0,
  fp16=True,
  report_to="none",
)

json.dump(args, open("train_llama3.1.json", "w", encoding="utf-8"), indent=2)

%cd /content/drive/MyDrive/5398/LLaMA-Factory
!llamafactory-cli train "/content/drive/MyDrive/5398/LLaMA-Factory/train_llama3.1.json"

/content/drive/MyDrive/5398/LLaMA-Factory
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-03-15 16:54:52] llamafactory.hparams.parser:144 >> Resuming training from llama3.1_lora/checkpoint-117.
[INFO|2026-03-15 16:54:52] llamafactory.hparams.parser:144 >> Change `output_dir` or use `overwrite_output_dir` to avoid.
[INFO|2026-03-15 16:54:52] llamafactory.hparams.parser:462 >> Process rank: 0, world size: 1, device: cuda:0, distributed training:

In [10]:
!python -m pip list | grep llamafactory

llamafactory                             0.9.5.dev0         /content/drive/MyDrive/5398/LLaMA-Factory


In [11]:
!which python

/usr/local/bin/python


In [24]:
%cd /content/drive/MyDrive/5398/LLaMA-Factory
!python -m pip install -e .
#after running this cell, restart the runtime. Do not rerun this cell after restarting

/content/drive/MyDrive/5398/LLaMA-Factory
Obtaining file:///content/drive/MyDrive/5398/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done
  Created wheel for llamafactory: filename=llamafactory-0.9.5.dev0-py3-none-any.whl size=26999 sha256=92f8249a907c6938beda7479f7d1135c5b098365aac07733fcd12af9bffb5a67
  Stored in directory: /tmp/pip-ephem-wheel-cache-s7avy0q_/wheels/4a/98/83/0ecf5c7c171d2c5f1089381f844e4c9bc1ff306066f8b26dff
Successfully built llamafactory
  Attempting uninstall: llamafactory
    Found existing installation: llamafactory 0.9.5.dev0
    Uninstalling llamafactory-0.9.5.dev0:
      Successfully uninstalled llamafactory-0.9.5.dev0


In [12]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc
args = dict(
  model_name_or_path="meta-llama/Llama-3.1-8B",
  adapter_name_or_path="llama3.1_lora",
  template="llama3",
  finetuning_type="lora",
  max_new_tokens = 512
)
chat_model = ChatModel(args)

[INFO|configuration_utils.py:667] 2026-03-15 16:57:49,947 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/config.json
[INFO|configuration_utils.py:739] 2026-03-15 16:57:49,950 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embe

[INFO|2026-03-15 16:57:56] llamafactory.data.template:144 >> Replace eos token: <|eot_id|>.
[INFO|2026-03-15 16:57:56] llamafactory.data.template:144 >> Add pad token: <|eot_id|>
[INFO|2026-03-15 16:57:56] llamafactory.data.template:144 >> Add <|eom_id|> to stop words.


[INFO|configuration_utils.py:667] 2026-03-15 16:57:56,949 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/config.json
[INFO|configuration_utils.py:739] 2026-03-15 16:57:56,950 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embe

[INFO|2026-03-15 16:57:56] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:732] 2026-03-15 16:57:59,407 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-15 16:57:59,409 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-15 16:57:59,410 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-15 16:57:59,718 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO|configuration_utils.py:967] 2026-03-15 16:58:05,055 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-15 16:58:05,056 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.9
}



[INFO|2026-03-15 16:58:05] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-15 16:58:06] llamafactory.model.adapter:144 >> Merged 1 adapter(s).
[INFO|2026-03-15 16:58:06] llamafactory.model.adapter:144 >> Loaded adapter(s): llama3.1_lora
[INFO|2026-03-15 16:58:06] llamafactory.model.loader:144 >> all params: 8,030,261,248


In [13]:
system_text = "You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:\n\n[Positive Developments]:\n1. ...\n\n[Potential Concerns]:\n1. ...\n\n[Prediction & Analysis]\nPrediction: ...\nAnalysis: ...\n\n"
messages = [
    {"role": "user", "content": system_text + "\n" +"[Company Introduction]:\n\nAmerican Express Co is a leading entity in the Financial Services sector. Incorporated and publicly traded since 1977-05-18, the company has established its reputation as one of the key players in the market. As of today, American Express Co has a market capitalization of 168338.49 in USD, with 723.87 shares outstanding.\n\nAmerican Express Co operates primarily in the US, trading under the ticker AXP on the NEW YORK STOCK EXCHANGE, INC.. As a dominant force in the Financial Services space, the company continues to innovate and drive progress within the industry.\n\nFrom 2024-02-11 to 2024-02-18, AXP's stock price increased from 211.81 to 211.90. News during this period are listed below:\n\n[Headline]: American Express: Buy, Sell, or Hold?\n[Summary]: With shares at all-time highs, investors still won't struggle in finding reasons to be bullish.\n\n[Headline]: American Express Co. stock underperforms Monday when compared to competitors\n[Summary]: Shares of American Express Co. slipped 0.10% to $212.26 Monday, on what proved to be an all-around mixed trading session for the stock market, with the Dow...\n\n[Headline]: The Next Berkshire Hathaway? 3 Blue-Chip Stocks That Investors Shouldn’t Ignore.\n[Summary]: Blue chip stocks are what many investors look for when trying to add new stocks to their portfolios. What really are these companies? Well, a blue chip company is generally a well-known and well-established company that has a history of generating consistent profits. They have grown their revenue in the past and have worked to make a successful company. That’s why these are usually more reliable to invest in since they are not small startup companies that just hit the market and carry tons of ri\n\n[Headline]: Tracking Mario Gabelli's Gabelli Funds 13F Portfolio - Q4 2023 Update\n[Summary]: Gabelli Funds' 13F portfolio value increased by approximately 5% in Q4 2023. Click here to read more about Mario Gabelli's holdings and trades for Q4 2023.\n\n[Headline]: American Express Co. stock underperforms Thursday when compared to competitors despite daily gains\n[Summary]: Shares of American Express Co. inched 0.77% higher to $212.53 Thursday, on what proved to be an all-around positive trading session for the stock market,...\n\nFrom 2024-02-18 to 2024-02-25, AXP's stock price increased from 211.90 to 213.90. News during this period are listed below:\n\n[Headline]: The Dow is due for a change. These sectors may be affected.\n[Summary]: The keepers of the Dow Jones Industrial Average will have to make at least one change to the venerable index next week, and there’s a very good chance...\n\n[Headline]: Discover Stock Jumps on Capital One Deal News\n[Summary]: Shares of Discover Financial Services jumped in early trading following the news that Capital One will buy the credit-card company for more than $35 billion.  Discover's shares were up 14% in recent trading, while Capital One's rose 1%.  Capital One stock was up more than 4%.\n\n[Headline]: Got $1,000? 3 Payments Stocks to Buy and Hold Forever\n[Summary]: Americans spend trillions of dollars on debit and credit cards; these three companies are behind almost every transaction.\n\n[Headline]: Are You a Growth Investor? This 1 Stock Could Be the Perfect Pick\n[Summary]: The Zacks Style Scores offers investors a way to easily find top-rated stocks based on their investing style. Here's why you should take advantage.\n\n[Headline]: Nearly One-Fifth of Warren Buffett-Led Berkshire Hathaway's $368 Billion Stock Portfolio Is Invested in These 2 Financial Giants\n[Summary]: These two industry heavyweights should be on your investing radar.\n\nSome recent basic financials of AXP, reported at 2023-12-31, are presented below:\n\n[Basic Financials]:\n\nassetTurnoverTTM: 0.2635\nbookValue: 28057\ncashRatio: 0.31986958884260097\ncurrentRatio: 0.7395\nebitPerShare: 3.4553\neps: 2.6589\nev: 268910.19\nfcfMargin: 0.3691\nfcfPerShareTTM: 23.5076\ngrossMargin: 0.6334\nlongtermDebtTotalAsset: 0.1833\nlongtermDebtTotalCapital: 0.232\nlongtermDebtTotalEquity: 1.706\nnetDebtToTotalCapital: 0.6415\nnetDebtToTotalEquity: 4.7185\nnetMargin: 0.1125\noperatingMargin: 0.1462\npayoutRatioTTM: 0.2126\npb: 4.8659\npeTTM: 16.3032\npfcfTTM: 7.6988\npretaxMargin: 0.1462\npsTTM: 2.0881\nquickRatio: 0.7395\nreceivablesTurnoverTTM: 1.1117\nroaTTM: 0.0338\nroeTTM: 0.3099\nroicTTM: 0.0422\nrotcTTM: 0.053\nsalesPerShare: 23.6369\nsgaToSale: 0.3666\ntotalDebtToEquity: 6.355\ntotalDebtToTotalAsset: 0.6829\ntotalDebtToTotalCapital: 0.864\ntotalRatio: 1.1204\n\nBased on all the information before 2024-02-25, let's first analyze the positive developments and potential concerns for AXP. Come up with 2-4 most important factors respectively and keep them concise. Most factors should be inferred from company related news. Then make your prediction of the AXP cryptocurrency price movement for next week (2024-02-25 to 2024-03-03). Provide a summary analysis to support your prediction."}
]

response = ""
for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text

torch_gc()

[Positive Developments]:
1. American Express Co. has seen its stock price consistently increasing over the past weeks, indicating a strong performance in the market.
2. AXP has been highlighted as a blue-chip stock and is recognized as a reliable investment due to its consistent profits and revenue growth.
3. The company's financials show a healthy gross margin of 0.6334, suggesting effective cost management and profitability.
4. American Express Co. has been identified as one of the key players in the payments industry, indicating its dominant position in the market.

[Potential Concerns]:
1. The company's stock underperformed in comparison to its competitors on some trading days, indicating potential weakness in its market position.
2. AXP's long-term debt to total equity ratio is quite high at 1.706, suggesting the company has been aggressive in financing its growth with debt.
3. The company's current ratio is less than 1 (0.7395), indicating potential issues with meeting short-term

In [14]:
#merge and saving model

args = dict(
  model_name_or_path="meta-llama/Llama-3.1-8B",
  adapter_name_or_path="llama3.1_lora",
  template="llama3",
  finetuning_type="lora",
  export_dir="llama3.1_lora_merged",
  export_size=2,
  export_device="auto",
  # export_hub_model_id="your_id/your_model",
)

json.dump(args, open("merge_llama3.1.json", "w", encoding="utf-8"), indent=2)

%cd /content/drive/MyDrive/5398/LLaMA-Factory

!llamafactory-cli export merge_llama3.1.json

/content/drive/MyDrive/5398/LLaMA-Factory
[INFO|configuration_utils.py:667] 2026-03-15 16:59:25,700 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/config.json
[INFO|configuration_utils.py:739] 2026-03-15 16:59:25,702 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_fac

In [15]:
df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_system.json")
print(df_test.columns)

Index(['instruction', 'input', 'output', 'system'], dtype='object')


**1.2 Testing Finetuned Model**

In [16]:
args = dict(
    model_name_or_path="llama3.1_lora_merged",
    template="llama3",
    finetuning_type="lora",
    max_new_tokens = 512
)
chat_model = ChatModel(args)

df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_system.json")
# column: 'instruction' -> model input, 'output' -> correct answer

rouge = Rouge()
results = []

for idx, row in df_test.iterrows():
    system_text = row["system"] if pd.notna(row["system"]) else ""

    user_content = row["instruction"]
    if pd.notna(row.get("input", "")) and row.get("input", "") != "":
      user_content += "\n" + row["input"]

    full_prompt = system_text + "\n" + user_content

    messages = [
        {"role": "user", "content": full_prompt}
    ]

    start_time = time.time()

    response = ""
    for new_text in chat_model.stream_chat(messages):
        response += new_text

    output = response

    inference_time = time.time() - start_time

    scores = rouge.get_scores(output, row["output"])[0]

    results.append({
        "prompt": row["instruction"],
        "system": row.get("system", ""),
        "reference": row["output"],
        "prediction": output,
        "rouge-1_f": scores['rouge-1']['f'],
        "rouge-2_f": scores['rouge-2']['f'],
        "rouge-l_f": scores['rouge-l']['f'],
        "inference_time_s": inference_time
    })

    torch_gc()


df_results = pd.DataFrame(results)
df_results.to_csv("/content/drive/MyDrive/5398/LLaMA-Factory/llama3.1_test_results.csv", index=False)

print("save llama3.1_test_results.csv")

[INFO|configuration_utils.py:665] 2026-03-15 17:09:14,543 >> loading configuration file llama3.1_lora_merged/config.json
[INFO|configuration_utils.py:739] 2026-03-15 17:09:14,544 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transform

[INFO|2026-03-15 17:09:18] llamafactory.data.template:144 >> Add <|eom_id|> to stop words.


[INFO|configuration_utils.py:665] 2026-03-15 17:09:18,474 >> loading configuration file llama3.1_lora_merged/config.json
[INFO|configuration_utils.py:739] 2026-03-15 17:09:18,475 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transform

[INFO|2026-03-15 17:09:18] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:729] 2026-03-15 17:09:18,479 >> loading weights file llama3.1_lora_merged/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-15 17:09:18,481 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-15 17:09:18,483 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-15 17:09:18,529 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO|configuration_utils.py:965] 2026-03-15 17:09:28,749 >> loading configuration file llama3.1_lora_merged/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-15 17:09:28,750 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.9
}



[INFO|2026-03-15 17:09:28] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-15 17:09:28] llamafactory.model.loader:144 >> all params: 8,030,261,248
save llama3.1_test_results.csv


**1.3 Testing Base Model**

In [17]:
args = dict(
    model_name_or_path="meta-llama/Llama-3.1-8B",
    template="llama3",
    max_new_tokens=512
)
chat_model = ChatModel(args)

df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_system.json")


rouge = Rouge()
results = []

for idx, row in df_test.iterrows():
    system_text = row["system"] if pd.notna(row["system"]) else ""

    user_content = row["instruction"]
    if pd.notna(row.get("input", "")) and row.get("input", "") != "":
        user_content += "\n" + row["input"]

    full_prompt = system_text + "\n" + user_content

    messages = [
        {"role": "user", "content": full_prompt}
    ]

    start_time = time.time()

    response = ""
    for new_text in chat_model.stream_chat(messages):
        response += new_text

    output = response
    inference_time = time.time() - start_time

    scores = rouge.get_scores(output, row["output"])[0]

    results.append({
        "prompt": row["instruction"],
        "system": row.get("system", ""),
        "reference": row["output"],
        "prediction": output,
        "rouge-1_f": scores['rouge-1']['f'],
        "rouge-2_f": scores['rouge-2']['f'],
        "rouge-l_f": scores['rouge-l']['f'],
        "inference_time_s": inference_time
    })

    torch_gc()

df_results = pd.DataFrame(results)
df_results.to_csv("/content/drive/MyDrive/5398/LLaMA-Factory/llama3.1_base_test_results.csv", index=False)

print("save llama3.1_base_test_results.csv")

[INFO|configuration_utils.py:667] 2026-03-15 18:48:03,474 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/config.json
[INFO|configuration_utils.py:739] 2026-03-15 18:48:03,475 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embe

[INFO|2026-03-15 18:48:10] llamafactory.data.template:144 >> Replace eos token: <|eot_id|>.
[INFO|2026-03-15 18:48:10] llamafactory.data.template:144 >> Add pad token: <|eot_id|>
[INFO|2026-03-15 18:48:10] llamafactory.data.template:144 >> Add <|eom_id|> to stop words.


[INFO|configuration_utils.py:667] 2026-03-15 18:48:10,741 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/config.json
[INFO|configuration_utils.py:739] 2026-03-15 18:48:10,743 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embe

[INFO|2026-03-15 18:48:10] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:732] 2026-03-15 18:48:10,989 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-15 18:48:10,991 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-15 18:48:10,992 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-15 18:48:11,040 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO|configuration_utils.py:967] 2026-03-15 18:48:15,765 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-15 18:48:15,766 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.9
}



[INFO|2026-03-15 18:48:15] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-15 18:48:15] llamafactory.model.loader:144 >> all params: 8,030,261,248
save llama3.1_base_test_results.csv


# **2. Deepseek R1**

**2.1 Fine Tune**

In [18]:
args = dict(
  stage="sft",
  do_train=True,
  model_name_or_path="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
  dataset="df_train_llama",
  template="deepseekr1",
  finetuning_type="lora",
  lora_target="q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
  output_dir="DeepSeek-R1_lora",
  per_device_train_batch_size=2,
  gradient_accumulation_steps=16,
  lr_scheduler_type="cosine",
  logging_steps=5,
  warmup_ratio=0.1,
  save_steps=1000,
  learning_rate=5e-5,
  num_train_epochs=3.0,
  max_grad_norm=1.0,
  loraplus_lr_ratio=16.0,
  fp16=True,
  report_to="none",
)

json.dump(args, open("train_deepseekR1.json", "w", encoding="utf-8"), indent=2)

%cd /content/drive/MyDrive/5398/LLaMA-Factory
!llamafactory-cli train "/content/drive/MyDrive/5398/LLaMA-Factory/train_deepseekR1.json"

/content/drive/MyDrive/5398/LLaMA-Factory
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-03-15 20:26:16] llamafactory.hparams.parser:144 >> Resuming training from DeepSeek-R1_lora/checkpoint-117.
[INFO|2026-03-15 20:26:16] llamafactory.hparams.parser:144 >> Change `output_dir` or use `overwrite_output_dir` to avoid.
[INFO|2026-03-15 20:26:16] llamafactory.hparams.parser:462 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
config.json: 100% 826/826 [00:00<00:00, 3.40MB/s]
[INFO|configuration_utils.py:667] 2026-03-15 20:26:17,325 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/config.json
[INFO|configuration_utils.py:739] 2026-03-15 20:26:17,326 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": fal

In [19]:
args = dict(
  model_name_or_path="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
  adapter_name_or_path="DeepSeek-R1_lora",
  template="deepseekr1",
  finetuning_type="lora",
)
chat_model = ChatModel(args)

[INFO|configuration_utils.py:667] 2026-03-15 20:27:33,668 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/config.json
[INFO|configuration_utils.py:739] 2026-03-15 20:27:33,670 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_

[WARNING|2026-03-15 20:27:40] llamafactory.data.template:149 >> You are using reasoning template, please add `_nothink` suffix if the model is not a reasoning model. e.g., qwen3_vl_nothink


[INFO|configuration_utils.py:667] 2026-03-15 20:27:41,219 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/config.json
[INFO|configuration_utils.py:739] 2026-03-15 20:27:41,220 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_

[INFO|2026-03-15 20:27:41] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:732] 2026-03-15 20:27:41,470 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-15 20:27:41,472 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-15 20:27:41,473 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-15 20:27:41,522 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

[INFO|configuration_utils.py:967] 2026-03-15 20:27:47,167 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-15 20:27:47,168 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.95
}



[INFO|2026-03-15 20:27:47] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-15 20:27:55] llamafactory.model.adapter:144 >> Merged 1 adapter(s).
[INFO|2026-03-15 20:27:55] llamafactory.model.adapter:144 >> Loaded adapter(s): DeepSeek-R1_lora
[INFO|2026-03-15 20:27:55] llamafactory.model.loader:144 >> all params: 8,030,261,248


In [20]:
messages = [
    {"role": "user", "content": '''You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:

[Positive Developments]:
1. ...

[Potential Concerns]:
1. ...

[Prediction & Analysis]
Prediction: ...
Analysis: ...

<</SYS>>

[Company Introduction]:

American Express Co is a leading entity in the Financial Services sector. Incorporated and publicly traded since 1977-05-18, the company has established its reputation as one of the key players in the market. As of today, American Express Co has a market capitalization of 168338.49 in USD, with 723.87 shares outstanding.

American Express Co operates primarily in the US, trading under the ticker AXP on the NEW YORK STOCK EXCHANGE, INC.. As a dominant force in the Financial Services space, the company continues to innovate and drive progress within the industry.

From 2023-05-07 to 2023-05-14, AXP's stock price decreased from 150.55 to 145.90. News during this period are listed below:

[Headline]: Credit cards: issuers could use their own flexible friends
[Summary]: Concerns profitability has peaked following a strong 2022 are not baseless, but even so credit cards remain a lucrative business

[Headline]: Dow's nearly 175-point fall led by losses in American Express, Nike shares
[Summary]: Dragged down by losses for shares of American Express and Nike, the Dow Jones Industrial Average is trading down Wednesday afternoon. Shares of American...

[Headline]: Chase is opening its first U.S. airport lounge in Boston — here's how you can get in
[Summary]: The new Chase Sapphire Lounge at Boston Airport features complimentary food, drinks, rest pods, massage chairs and more. Here's how you can get in.

[Headline]: American Express Co. stock outperforms competitors despite losses on the day
[Summary]: Shares of American Express Co. sank 0.04% to $147.93 Friday, on what proved to be an all-around dismal trading session for the stock market, with the S&P 500...

[Headline]: How To Build A Dividend Portfolio With $25,000 Among May's Top 30 Stocks
[Summary]: An investment strategy that combines dividend income with dividend growth, brings several benefits for dividend income investors. Click here to read more.

Some recent basic financials of AXP, reported at 2023-03-31, are presented below:

[Basic Financials]:

assetTurnoverTTM: 0.2614
bookValue: 25992
cashRatio: 0.2944926548987087
currentRatio: 0.7185
ebitPerShare: 2.9126
eps: 2.4409
ev: 246452.85
fcfPerShareTTM: 20.2611
grossMargin: 0.624
longtermDebtTotalAsset: 0.1744
longtermDebtTotalCapital: 0.2169
longtermDebtTotalEquity: 1.5827
netDebtToTotalCapital: 0.6532
netDebtToTotalEquity: 4.7667
netMargin: 0.1189
operatingMargin: 0.1419
payoutRatioTTM: 0.2247
pb: 4.7152
peTTM: 16.9489
pretaxMargin: 0.1419
psTTM: 2.1201
quickRatio: 0.7185
receivablesTurnoverTTM: 1.0476
roaTTM: 0.0327
roeTTM: 0.2955
roicTTM: 0.0412
rotcTTM: 0.0515
salesPerShare: 20.5309
sgaToSale: 0.376
totalDebtToEquity: 6.2969
totalDebtToTotalAsset: 0.694
totalDebtToTotalCapital: 0.863
totalRatio: 1.1239

Based on all the information before 2023-05-14, let's first analyze the positive developments and potential concerns for AXP. Come up with 2-4 most important factors respectively and keep them concise. Most factors should be inferred from company related news. Then make your prediction of the AXP cryptocurrency price movement for next week (2023-05-14 to 2023-05-21). Provide a summary analysis to support your prediction.'''}
]

response = ""
for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text

torch_gc()

<think>

</think>

[Positive Developments]:
1. American Express has outperformed its competitors despite some losses in the stock market, which suggests the company's resilience and strong market position.
2. The Chase Sapphire Lounge opening at Boston Airport, a new initiative by American Express, indicates their continuous effort to enhance customer experience and attract more clients, potentially increasing revenue.

[Potential Concerns]:
1. Concerns about profitability peaking following a strong 2022 have been raised, which might make investors cautious about the company's future growth prospects.
2. The recent decrease in the company's stock price from $150.55 to $145.90 in a week could be a sign of market sentiment turning negative towards American Express.
3. The company's total debt to equity ratio of 6.2969 and net debt to total equity ratio of 4.7667 indicates that the company is highly leveraged, which could pose a risk in case of a downturn in the economy or if the company'

In [21]:
args = dict(
  model_name_or_path="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
  adapter_name_or_path="DeepSeek-R1_lora",
  template="deepseekr1",
  finetuning_type="lora",
  export_dir="DeepseekR1_lora_merged",
  export_size=2,
  export_device="auto",
  # export_hub_model_id="your_id/your_model",
)

json.dump(args, open("merge_DeepseekR1.json", "w", encoding="utf-8"), indent=2)

%cd /content/drive/MyDrive/5398/LLaMA-Factory

!llamafactory-cli export merge_DeepseekR1.json

/content/drive/MyDrive/5398/LLaMA-Factory
[INFO|configuration_utils.py:667] 2026-03-15 20:28:25,610 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/config.json
[INFO|configuration_utils.py:739] 2026-03-15 20:28:25,612 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,


In [22]:
df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_llama.json")
print(df_test.columns)

Index(['instruction', 'input', 'output'], dtype='object')


**2.2 Testing Finetuned Model**

In [23]:
args = dict(
    model_name_or_path="DeepseekR1_lora_merged",
    template="deepseekr1",
    finetuning_type="lora"
)
chat_model = ChatModel(args)

df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_llama.json")


rouge = Rouge()
results = []


for idx, row in df_test.iterrows():
    messages = [{"role": "user", "content": row["instruction"]}]

    # 计时
    start_time = time.time()
    output = "".join(chat_model.stream_chat(messages))
    inference_time = time.time() - start_time

    scores = rouge.get_scores(output, row["output"])[0]

    results.append({
        "prompt": row["instruction"],
        "reference": row["output"],
        "prediction": output,
        "rouge-1_f": scores['rouge-1']['f'],
        "rouge-2_f": scores['rouge-2']['f'],
        "rouge-l_f": scores['rouge-l']['f'],
        "inference_time_s": inference_time
    })

    torch_gc()


df_results = pd.DataFrame(results)
df_results.to_csv("/content/drive/MyDrive/5398/LLaMA-Factory/test_results.csv", index=False)

print("save test_results.csv")

[INFO|configuration_utils.py:665] 2026-03-15 20:36:30,405 >> loading configuration file DeepseekR1_lora_merged/config.json
[INFO|configuration_utils.py:739] 2026-03-15 20:36:30,406 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transfo

[WARNING|2026-03-15 20:36:43] llamafactory.data.template:149 >> You are using reasoning template, please add `_nothink` suffix if the model is not a reasoning model. e.g., qwen3_vl_nothink


[INFO|configuration_utils.py:665] 2026-03-15 20:36:43,776 >> loading configuration file DeepseekR1_lora_merged/config.json
[INFO|configuration_utils.py:739] 2026-03-15 20:36:43,777 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transfo

[INFO|2026-03-15 20:36:43] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:729] 2026-03-15 20:36:43,781 >> loading weights file DeepseekR1_lora_merged/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-15 20:36:43,784 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-15 20:36:43,785 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-15 20:36:43,831 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO|configuration_utils.py:965] 2026-03-15 20:37:24,183 >> loading configuration file DeepseekR1_lora_merged/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-15 20:37:24,184 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.95
}



[INFO|2026-03-15 20:37:24] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-15 20:37:24] llamafactory.model.loader:144 >> all params: 8,030,261,248
save test_results.csv


**2.3 Testing Base Model**

In [24]:
args = dict(
    model_name_or_path="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    template="deepseekr1"
)
chat_model = ChatModel(args)

df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_llama.json")

rouge = Rouge()
results = []

for idx, row in df_test.iterrows():
    messages = [{"role": "user", "content": row["instruction"]}]

    # 计时
    start_time = time.time()
    output = "".join(chat_model.stream_chat(messages))
    inference_time = time.time() - start_time

    scores = rouge.get_scores(output, row["output"])[0]

    results.append({
        "prompt": row["instruction"],
        "reference": row["output"],
        "prediction": output,
        "rouge-1_f": scores['rouge-1']['f'],
        "rouge-2_f": scores['rouge-2']['f'],
        "rouge-l_f": scores['rouge-l']['f'],
        "inference_time_s": inference_time
    })

    torch_gc()

df_results = pd.DataFrame(results)
df_results.to_csv("/content/drive/MyDrive/5398/LLaMA-Factory/Deepseek_base_test_results.csv", index=False)

print("save Deepseek_base_test_results.csv")

[INFO|configuration_utils.py:667] 2026-03-15 21:49:39,575 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/config.json
[INFO|configuration_utils.py:739] 2026-03-15 21:49:39,576 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_

[WARNING|2026-03-15 21:49:46] llamafactory.data.template:149 >> You are using reasoning template, please add `_nothink` suffix if the model is not a reasoning model. e.g., qwen3_vl_nothink


[INFO|configuration_utils.py:667] 2026-03-15 21:49:47,088 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/config.json
[INFO|configuration_utils.py:739] 2026-03-15 21:49:47,090 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128001,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_

[INFO|2026-03-15 21:49:47] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:732] 2026-03-15 21:49:47,342 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-15 21:49:47,344 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-15 21:49:47,345 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-15 21:49:47,392 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[INFO|configuration_utils.py:967] 2026-03-15 21:50:16,764 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Llama-8B/snapshots/6a6f4aa4197940add57724a7707d069478df56b1/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-15 21:50:16,765 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": 128001,
  "temperature": 0.6,
  "top_p": 0.95
}



[INFO|2026-03-15 21:50:16] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-15 21:50:16] llamafactory.model.loader:144 >> all params: 8,030,261,248
save Deepseek_base_test_results.csv


# **3. Qwen 2.5 7b Instruct**

**3.1 Fine Tune**

In [25]:
args = dict(
  stage="sft",
  do_train=True,
  model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
  dataset="df_train_system",
  template="qwen",
  finetuning_type="lora",
  lora_target="q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",
  output_dir="Qwen2.5_lora",
  per_device_train_batch_size=2,
  gradient_accumulation_steps=16,
  lr_scheduler_type="cosine",
  logging_steps=5,
  warmup_ratio=0.1,
  save_steps=1000,
  learning_rate=5e-5,
  num_train_epochs=3.0,
  max_grad_norm=1.0,
  loraplus_lr_ratio=16.0,
  fp16=True,
  report_to="none",
)

json.dump(args, open("train_Qwen2.5.json", "w", encoding="utf-8"), indent=2)

%cd /content/drive/MyDrive/5398/LLaMA-Factory
!llamafactory-cli train "/content/drive/MyDrive/5398/LLaMA-Factory/train_Qwen2.5.json"

/content/drive/MyDrive/5398/LLaMA-Factory
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-03-16 00:30:45] llamafactory.hparams.parser:144 >> Resuming training from Qwen2.5_lora/checkpoint-117.
[INFO|2026-03-16 00:30:45] llamafactory.hparams.parser:144 >> Change `output_dir` or use `overwrite_output_dir` to avoid.
[INFO|2026-03-16 00:30:45] llamafactory.hparams.parser:462 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
config.json: 100% 663/663 [00:00<00:00, 3.69MB/s]
[INFO|configuration_utils.py:667] 2026-03-16 00:30:46,228 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/config.json
[INFO|configuration_utils.py:739] 2026-03-16 00:30:46,230 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id

In [26]:
args = dict(
  model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
  adapter_name_or_path="Qwen2.5_lora",
  template="qwen",
  finetuning_type="lora"
)
chat_model = ChatModel(args)


[INFO|configuration_utils.py:667] 2026-03-16 00:32:05,159 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/config.json
[INFO|configuration_utils.py:739] 2026-03-16 00:32:05,160 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",

[INFO|2026-03-16 00:32:10] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:732] 2026-03-16 00:32:11,117 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-16 00:32:11,119 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-16 00:32:11,120 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-16 00:32:11,168 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[INFO|configuration_utils.py:967] 2026-03-16 00:32:35,659 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-16 00:32:35,660 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



[INFO|2026-03-16 00:32:35] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-16 00:32:44] llamafactory.model.adapter:144 >> Merged 1 adapter(s).
[INFO|2026-03-16 00:32:44] llamafactory.model.adapter:144 >> Loaded adapter(s): Qwen2.5_lora
[INFO|2026-03-16 00:32:44] llamafactory.model.loader:144 >> all params: 7,615,616,512


In [27]:
system_text = "You are a seasoned stock market analyst. Your task is to list the positive developments and potential concerns for companies based on relevant news and basic financials from the past weeks, then provide an analysis and prediction for the companies' stock price movement for the upcoming week. Your answer format should be as follows:\n\n[Positive Developments]:\n1. ...\n\n[Potential Concerns]:\n1. ...\n\n[Prediction & Analysis]\nPrediction: ...\nAnalysis: ...\n\n"
messages = [
    {"role": "user", "content": system_text + "\n" +"[Company Introduction]:\n\nAmerican Express Co is a leading entity in the Financial Services sector. Incorporated and publicly traded since 1977-05-18, the company has established its reputation as one of the key players in the market. As of today, American Express Co has a market capitalization of 168338.49 in USD, with 723.87 shares outstanding.\n\nAmerican Express Co operates primarily in the US, trading under the ticker AXP on the NEW YORK STOCK EXCHANGE, INC.. As a dominant force in the Financial Services space, the company continues to innovate and drive progress within the industry.\n\nFrom 2024-02-11 to 2024-02-18, AXP's stock price increased from 211.81 to 211.90. News during this period are listed below:\n\n[Headline]: American Express: Buy, Sell, or Hold?\n[Summary]: With shares at all-time highs, investors still won't struggle in finding reasons to be bullish.\n\n[Headline]: American Express Co. stock underperforms Monday when compared to competitors\n[Summary]: Shares of American Express Co. slipped 0.10% to $212.26 Monday, on what proved to be an all-around mixed trading session for the stock market, with the Dow...\n\n[Headline]: The Next Berkshire Hathaway? 3 Blue-Chip Stocks That Investors Shouldn’t Ignore.\n[Summary]: Blue chip stocks are what many investors look for when trying to add new stocks to their portfolios. What really are these companies? Well, a blue chip company is generally a well-known and well-established company that has a history of generating consistent profits. They have grown their revenue in the past and have worked to make a successful company. That’s why these are usually more reliable to invest in since they are not small startup companies that just hit the market and carry tons of ri\n\n[Headline]: Tracking Mario Gabelli's Gabelli Funds 13F Portfolio - Q4 2023 Update\n[Summary]: Gabelli Funds' 13F portfolio value increased by approximately 5% in Q4 2023. Click here to read more about Mario Gabelli's holdings and trades for Q4 2023.\n\n[Headline]: American Express Co. stock underperforms Thursday when compared to competitors despite daily gains\n[Summary]: Shares of American Express Co. inched 0.77% higher to $212.53 Thursday, on what proved to be an all-around positive trading session for the stock market,...\n\nFrom 2024-02-18 to 2024-02-25, AXP's stock price increased from 211.90 to 213.90. News during this period are listed below:\n\n[Headline]: The Dow is due for a change. These sectors may be affected.\n[Summary]: The keepers of the Dow Jones Industrial Average will have to make at least one change to the venerable index next week, and there’s a very good chance...\n\n[Headline]: Discover Stock Jumps on Capital One Deal News\n[Summary]: Shares of Discover Financial Services jumped in early trading following the news that Capital One will buy the credit-card company for more than $35 billion.  Discover's shares were up 14% in recent trading, while Capital One's rose 1%.  Capital One stock was up more than 4%.\n\n[Headline]: Got $1,000? 3 Payments Stocks to Buy and Hold Forever\n[Summary]: Americans spend trillions of dollars on debit and credit cards; these three companies are behind almost every transaction.\n\n[Headline]: Are You a Growth Investor? This 1 Stock Could Be the Perfect Pick\n[Summary]: The Zacks Style Scores offers investors a way to easily find top-rated stocks based on their investing style. Here's why you should take advantage.\n\n[Headline]: Nearly One-Fifth of Warren Buffett-Led Berkshire Hathaway's $368 Billion Stock Portfolio Is Invested in These 2 Financial Giants\n[Summary]: These two industry heavyweights should be on your investing radar.\n\nSome recent basic financials of AXP, reported at 2023-12-31, are presented below:\n\n[Basic Financials]:\n\nassetTurnoverTTM: 0.2635\nbookValue: 28057\ncashRatio: 0.31986958884260097\ncurrentRatio: 0.7395\nebitPerShare: 3.4553\neps: 2.6589\nev: 268910.19\nfcfMargin: 0.3691\nfcfPerShareTTM: 23.5076\ngrossMargin: 0.6334\nlongtermDebtTotalAsset: 0.1833\nlongtermDebtTotalCapital: 0.232\nlongtermDebtTotalEquity: 1.706\nnetDebtToTotalCapital: 0.6415\nnetDebtToTotalEquity: 4.7185\nnetMargin: 0.1125\noperatingMargin: 0.1462\npayoutRatioTTM: 0.2126\npb: 4.8659\npeTTM: 16.3032\npfcfTTM: 7.6988\npretaxMargin: 0.1462\npsTTM: 2.0881\nquickRatio: 0.7395\nreceivablesTurnoverTTM: 1.1117\nroaTTM: 0.0338\nroeTTM: 0.3099\nroicTTM: 0.0422\nrotcTTM: 0.053\nsalesPerShare: 23.6369\nsgaToSale: 0.3666\ntotalDebtToEquity: 6.355\ntotalDebtToTotalAsset: 0.6829\ntotalDebtToTotalCapital: 0.864\ntotalRatio: 1.1204\n\nBased on all the information before 2024-02-25, let's first analyze the positive developments and potential concerns for AXP. Come up with 2-4 most important factors respectively and keep them concise. Most factors should be inferred from company related news. Then make your prediction of the AXP cryptocurrency price movement for next week (2024-02-25 to 2024-03-03). Provide a summary analysis to support your prediction."}
]

response = ""
for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text

torch_gc()

[Positive Developments]:
1. American Express Co. is listed as a blue-chip stock, which indicates its reliability and stability in the market.
2. The company's portfolio value increased by approximately 5% in Q4 2023, reflecting investor confidence in the company.
3. The company's free cash flow margin is relatively high at 0.3691, indicating a strong ability to generate cash.
4. The company's gross margin is at 0.6334, which suggests that it is efficient at converting sales into profits.

[Potential Concerns]:
1. The company's long-term debt to total asset ratio is 0.1833, indicating that the company has a relatively high level of debt.
2. The company's total debt to equity ratio is 6.355, which indicates that the company is highly leveraged.
3. The company's quick ratio is 0.7395, which is below 1, suggesting that the company may have difficulty meeting its short-term obligations.

[Prediction & Analysis]:
Prediction: Up by 2-3%
Analysis: Despite the potential concerns, the positive d

In [29]:
args = dict(
  model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
  adapter_name_or_path="Qwen2.5_lora",
  template="qwen",
  finetuning_type="lora",
  export_dir="Qwen2.5_lora_merged",
  export_size=2,
  export_device="auto",
  # export_hub_model_id="your_id/your_model",
)
json.dump(args, open("merge_Qwen2.5.json", "w", encoding="utf-8"), indent=2)

%cd /content/drive/MyDrive/5398/LLaMA-Factory

!llamafactory-cli export merge_Qwen2.5.json

/content/drive/MyDrive/5398/LLaMA-Factory
[INFO|configuration_utils.py:667] 2026-03-16 00:37:14,315 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/config.json
[INFO|configuration_utils.py:739] 2026-03-16 00:37:14,318 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
 

In [30]:
df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_system.json")
print(df_test.columns)

Index(['instruction', 'input', 'output', 'system'], dtype='object')


**3.2 Testing Finetuned Model**

In [31]:
args = dict(
    model_name_or_path="Qwen2.5_lora_merged",
    template="qwen",
    finetuning_type="lora",
)
chat_model = ChatModel(args)

df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_system.json")

rouge = Rouge()
results = []

for idx, row in df_test.iterrows():
    system_text = row["system"] if pd.notna(row["system"]) else ""

    user_content = row["instruction"]
    if pd.notna(row.get("input", "")) and row.get("input", "") != "":
      user_content += "\n" + row["input"]

    full_prompt = system_text + "\n" + user_content

    messages = [
        {"role": "user", "content": full_prompt}
    ]

    start_time = time.time()

    response = ""
    for new_text in chat_model.stream_chat(messages):
        response += new_text

    output = response

    inference_time = time.time() - start_time

    scores = rouge.get_scores(output, row["output"])[0]

    results.append({
        "prompt": row["instruction"],
        "system": row.get("system", ""),
        "reference": row["output"],
        "prediction": output,
        "rouge-1_f": scores['rouge-1']['f'],
        "rouge-2_f": scores['rouge-2']['f'],
        "rouge-l_f": scores['rouge-l']['f'],
        "inference_time_s": inference_time
    })

    torch_gc()

df_results = pd.DataFrame(results)
df_results.to_csv("/content/drive/MyDrive/5398/LLaMA-Factory/Qwen2.5_test_results.csv", index=False)

print("save Qwen2.5_test_results.csv")


[INFO|configuration_utils.py:665] 2026-03-16 00:47:07,357 >> loading configuration file Qwen2.5_lora_merged/config.json
[INFO|configuration_utils.py:739] 2026-03-16 00:47:07,359 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full

[INFO|2026-03-16 00:47:10] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:729] 2026-03-16 00:47:10,645 >> loading weights file Qwen2.5_lora_merged/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-16 00:47:10,647 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-16 00:47:10,651 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-16 00:47:10,699 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[INFO|configuration_utils.py:965] 2026-03-16 00:47:39,177 >> loading configuration file Qwen2.5_lora_merged/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-16 00:47:39,178 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



[INFO|2026-03-16 00:47:39] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-16 00:47:39] llamafactory.model.loader:144 >> all params: 7,615,616,512
save Qwen2.5_test_results.csv


**3.3 Testing base model**

In [32]:
args = dict(
    model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
    template="qwen",
)
chat_model = ChatModel(args)

df_test = pd.read_json("/content/drive/MyDrive/5398/LLaMA-Factory/data/df_test_system.json")


rouge = Rouge()
results = []


for idx, row in df_test.iterrows():
    system_text = row["system"] if pd.notna(row["system"]) else ""

    user_content = row["instruction"]
    if pd.notna(row.get("input", "")) and row.get("input", "") != "":
        user_content += "\n" + row["input"]

    full_prompt = system_text + "\n" + user_content

    messages = [
        {"role": "user", "content": full_prompt}
    ]

    start_time = time.time()

    response = ""
    for new_text in chat_model.stream_chat(messages):
        response += new_text

    output = response
    inference_time = time.time() - start_time

    scores = rouge.get_scores(output, row["output"])[0]

    results.append({
        "prompt": row["instruction"],
        "system": row.get("system", ""),
        "reference": row["output"],
        "prediction": output,
        "rouge-1_f": scores['rouge-1']['f'],
        "rouge-2_f": scores['rouge-2']['f'],
        "rouge-l_f": scores['rouge-l']['f'],
        "inference_time_s": inference_time
    })

    torch_gc()

df_results = pd.DataFrame(results)
df_results.to_csv("/content/drive/MyDrive/5398/LLaMA-Factory/Qwen2.5_base_test_results.csv", index=False)

print("save Qwen2.5_base_test_results.csv")

[INFO|configuration_utils.py:667] 2026-03-16 01:54:28,602 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/config.json
[INFO|configuration_utils.py:739] 2026-03-16 01:54:28,603 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 3584,
  "initializer_range": 0.02,
  "intermediate_size": 18944,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",

[INFO|2026-03-16 01:54:34] llamafactory.model.model_utils.kv_cache:144 >> KV cache is enabled for faster generation.


[INFO|modeling_utils.py:732] 2026-03-16 01:54:34,608 >> loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/model.safetensors.index.json
[INFO|modeling_utils.py:801] 2026-03-16 01:54:34,609 >> Will use dtype=torch.bfloat16 as defined in model's config object
[INFO|configuration_utils.py:1014] 2026-03-16 01:54:34,611 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

[INFO|accelerate.py:214] 2026-03-16 01:54:34,658 >> We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

[INFO|configuration_utils.py:967] 2026-03-16 01:54:39,469 >> loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28/generation_config.json
[INFO|configuration_utils.py:1014] 2026-03-16 01:54:39,470 >> Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



[INFO|2026-03-16 01:54:39] llamafactory.model.model_utils.attention:144 >> Using torch SDPA for faster training and inference.
[INFO|2026-03-16 01:54:39] llamafactory.model.loader:144 >> all params: 7,615,616,512
save Qwen2.5_base_test_results.csv
